# エラー検知 → 時間相関(±10分) → AI修正案 → メール送信

**前提**: Lakehouse `loghandson` をこのノートブックの既定Lakehouseにアタッチしてください。

各コードセルを上から順に実行します。`%pip install` は最初のセルで単独実行してください。

In [ ]:
# 依存関係のインストール(必ず最初のセル・単独で実行)
%pip install -q openai

In [ ]:
# 設定(ここだけ自分の環境に合わせて埋める)
# --- テーブル / 列名 ---
APP_TABLE     = "silver_app"      # 既定Lakehouseでない場合は "loghandson.silver_app"
ORACLE_TABLE  = "silver_oracle"
TS_COL        = "event_time"      # タイムスタンプ列(実テーブルの列名に合わせる)
LEVEL_COL     = "level"           # ログレベル列
MSG_COL       = "message"         # 本文列
WINDOW_MIN    = 10                # 前後の相関ウィンドウ(分)

# --- AIモデル ---
AI_MODEL      = "gpt-5-mini"      # 代替: "gpt-5.1" / "gpt-4.1"
API_VERSION   = "2025-04-01-preview"

# --- メール送信(作成後に自分で設定) ---
RECIPIENT_EMAIL = ""              # 例: "ichi@example.com"(複数は ; 区切り)
EMAIL_FLOW_URL  = ""              # Power Automate "Webhook受信時" フローのURL
# 両方が空のときは送信せず本文をprintするだけ(検知ロジックの単体テスト用)

In [ ]:
# (1) silver_app から最新のERROR 1件を抽出
from pyspark.sql import functions as F
from datetime import timedelta

app_df = (spark.table(APP_TABLE)
            .where(F.upper(F.col(LEVEL_COL)) == "ERROR")
            .orderBy(F.col(TS_COL).desc())
            .limit(1))

app_err = app_df.first()
if app_err is None:
    print("silver_app にERRORが見つかりませんでした。処理を終了します。")
    notebookutils.notebook.exit("NO_APP_ERROR")  # Fabric の正しい exit API

app_ts  = app_err[TS_COL]
app_msg = app_err[MSG_COL]
print(f"[App ERROR] {app_ts}\n{app_msg}\n")

In [ ]:
# (2) silver_oracle から ±WINDOW_MIN分のERRORを抽出
low  = app_ts - timedelta(minutes=WINDOW_MIN)
high = app_ts + timedelta(minutes=WINDOW_MIN)

ora_df = (spark.table(ORACLE_TABLE)
            .where(F.upper(F.col(LEVEL_COL)) == "ERROR")
            .where(F.col(TS_COL).between(F.lit(low), F.lit(high)))
            .orderBy(F.col(TS_COL).asc()))

ora_rows = ora_df.collect()
print(f"[Oracle ERROR in ±{WINDOW_MIN}min] {len(ora_rows)} 件")

if ora_rows:
    ora_block = "\n".join(f"- {r[TS_COL]}: {r[MSG_COL]}" for r in ora_rows)
else:
    ora_block = "(同時間帯に Oracle のエラーはありませんでした)"
print(ora_block + "\n")

In [ ]:
# (3) AIモデルに問い合わせて修正案を得る
from synapse.ml.fabric.credentials import get_openai_httpx_sync_client
import openai

client = openai.AzureOpenAI(
    http_client=get_openai_httpx_sync_client(),
    api_version=API_VERSION,
)

prompt = f"""あなたはシステム運用のエキスパートです。
以下のアプリケーションエラーと、その前後{WINDOW_MIN}分以内に発生したOracleのエラーを踏まえ、
(1)推定される原因 (2)具体的な修正手順 を日本語で簡潔に提案してください。

# アプリケーションエラー(発生時刻 {app_ts})
{app_msg}

# 同時間帯のOracleエラー
{ora_block}
"""

resp = client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": "簡潔で実践的な障害対応の提案を返す。"},
        {"role": "user",   "content": prompt},
    ],
)
suggestion = resp.choices[0].message.content
print("[AI 修正案]\n" + suggestion + "\n")

In [ ]:
# 本文(HTML)を組み立て
ora_html = ("<br>".join(f"・{r[TS_COL]}: {r[MSG_COL]}" for r in ora_rows)
            if ora_rows else "(同時間帯に Oracle のエラーはありませんでした)")

html_body = f"""<h3>エラー検知通知</h3>
<p><b>アプリケーションエラー</b>(発生時刻 {app_ts})<br>{app_msg}</p>
<p><b>同時間帯のOracleエラー(±{WINDOW_MIN}分)</b><br>{ora_html}</p>
<hr>
<p><b>AIによる修正案</b><br>{suggestion.replace(chr(10), '<br>')}</p>
"""
subject = f"[エラー検知] {app_ts} アプリエラー + Oracle {len(ora_rows)}件"

In [ ]:
# (4) メール送信(Power Automate 経由)
import requests

if RECIPIENT_EMAIL and EMAIL_FLOW_URL:
    payload = {"subject": subject, "to": RECIPIENT_EMAIL, "body": html_body}
    r = requests.post(EMAIL_FLOW_URL, json=payload, timeout=15)
    r.raise_for_status()
    print(f"メール送信OK → {RECIPIENT_EMAIL}")
else:
    print("メール未設定のため送信をスキップしました。以下が送信予定の本文です:\n")
    print("Subject:", subject)
    print(html_body)